Definição da Estrutura

In [0]:
from pyspark.sql import functions
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, ArrayType, DoubleType, DateType, LongType, TimestampType
 
df = spark.read.table("project_apis.bronze.districts_ibge_raw")

df_schema = ArrayType(
    StructType([
        StructField("id", StringType(), True),
        StructField("nome", StringType(), True),
        StructField("municipio", StructType([
            StructField("nome", StringType(), True),
            StructField("microrregiao", StructType([
                StructField("nome", StringType(), True),
                StructField("mesorregiao", StructType([
                    StructField("nome", StringType(), True),
                    StructField("UF", StructType([
                        StructField("nome", StringType(), True),
                        StructField("sigla", StringType(), True),
                        StructField("regiao", StructType([
                            StructField("nome", StringType(), True),
                            StructField("sigla", StringType(), True),
                        ]), True)
                    ]), True)
                ]), True)
            ]), True)
        ]), True)
    ])
)

print(df_schema)



Consolidação da Estrutura base com o Schema

In [0]:
df1 = df.withColumn("distritos_array",functions.from_json(functions.col("raw_json"), df_schema))
df1.show()
print(df1)
df1.printSchema

Tratamento da Estrutura e Criação da Tabela

In [0]:
df2 = df1.withColumn("distritos", functions.explode(functions.col("distritos_array")))
#df2.show()
#df2.printSchema()

df_split = df2.select(
    functions.col("distritos.id").cast(LongType()).alias("id"),
    functions.col("distritos.nome").cast(StringType()).alias("nome"),
    functions.col("distritos.municipio.nome").cast(StringType()).alias("nome_municipio"),
    functions.col("distritos.municipio.microrregiao.nome").cast(StringType()).alias("nome_microrregiao"),
    functions.col("distritos.municipio.microrregiao.mesorregiao.nome").cast(StringType()).alias("nome_mesorregiao"),
    functions.col("distritos.municipio.microrregiao.mesorregiao.UF.nome").cast(StringType()).alias("nome_UF"),
    functions.col("distritos.municipio.microrregiao.mesorregiao.UF.sigla").cast(StringType()).alias("sigla_UF"),
    functions.col("distritos.municipio.microrregiao.mesorregiao.UF.regiao.nome").cast(StringType()).alias("nome_regiao"),
    functions.col("distritos.municipio.microrregiao.mesorregiao.UF.regiao.sigla").cast(StringType()).alias("sigla_regiao"),
    functions.col("ingestion_timestamp").cast(TimestampType()),
    functions.col("source").cast(StringType())
)

df_split.show()
#df_split.printSchema()


Criação e Salvamento no Silver

In [0]:
# 1. Create a new Unity Catalog
#spark.sql("CREATE CATALOG IF NOT EXISTS my_catalog")

# 2. Create a new Schema (Database) inside that Catalog
#spark.sql("CREATE SCHEMA IF NOT EXISTS my_catalog.my_schema")

# 3. (Optional) Set the current context to this catalog and schema
#spark.sql("USE CATALOG my_catalog")
#spark.sql("USE SCHEMA my_schema")

# Informações Úteis

spark.sql("CREATE SCHEMA IF NOT EXISTS project_apis.silver")

df_split.write.format("delta").mode("overwrite").saveAsTable("project_apis.silver.districts_ibge")

result_sql = spark.sql("SELECT * FROM project_apis.silver.districts_ibge")
result_sql.show()

Aplicando Incremental

In [0]:
%sql
--ALTER TABLE project_apis.silver.districts_ibge ADD COLUMNS (created_at TIMESTAMP, updated_at TIMESTAMP)


In [0]:
result_sql = spark.sql("SELECT * FROM project_apis.silver.districts_ibge")
result_sql.show()

In [0]:
df_split.createOrReplaceTempView("staging_distritos")

In [0]:
spark.sql("""
MERGE INTO project_apis.silver.districts_ibge AS destino
USING staging_distritos AS origem
ON destino.id = origem.id

WHEN MATCHED THEN
  UPDATE SET
    destino.nome = origem.nome,
    destino.nome_municipio = origem.nome_municipio,
    destino.nome_microrregiao = origem.nome_microrregiao,
    destino.nome_mesorregiao = origem.nome_mesorregiao,
    destino.nome_UF = origem.nome_UF,
    destino.sigla_UF = origem.sigla_UF,
    destino.nome_regiao = origem.nome_regiao,
    destino.sigla_regiao = origem.sigla_regiao,
    destino.ingestion_timestamp = origem.ingestion_timestamp,
    destino.source = origem.source,
    destino.updated_at = current_timestamp()

WHEN NOT MATCHED THEN
  INSERT (
    id, nome, nome_municipio, nome_microrregiao, nome_mesorregiao,
    nome_UF, sigla_UF, nome_regiao, sigla_regiao,
    ingestion_timestamp, source, created_at, updated_at
  )
  VALUES (
    origem.id, origem.nome, origem.nome_municipio, origem.nome_microrregiao, origem.nome_mesorregiao,
    origem.nome_UF, origem.sigla_UF, origem.nome_regiao, origem.sigla_regiao,
    origem.ingestion_timestamp, origem.source, current_timestamp(), current_timestamp()
  )
""")